# CarRacing-v3 Deep Q-Network

This notebook will explore applying DQNs to the 'CarRacing-v3' gymnasium environment.

In [1]:
import random
import gymnasium as gym
import torch

from gymnasium import Space
from gymnasium.wrappers import RecordEpisodeStatistics, RecordVideo
from torch import nn
from torch.types import Number
from collections import deque, namedtuple
from itertools import count

In [2]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

## Deep Q-Networks

Regular Q-Networks use a table to store q-values, however when the state space is too large we cannot efficiently store and extract information from this table.

Deep Q-Networks use a deep neural network as a representation of this table, so we can pass in the state and get the approximated values q-values for all actions. We then use the highest q-valued action to select the best action.

In [3]:
class DQN(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            # Input (B, 3, 96, 96)
            nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            # Input (B, 64, 48, 48)
            nn.Conv2d(64, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            # Input (B, 32, 24, 24)
            nn.Conv2d(32, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            # Input (B, 16, 12, 12)
            nn.Conv2d(16, 8, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            # (B, 8, 6, 6)
            nn.Flatten(
                start_dim=1,
            ),
            # Input (B, 288)
            nn.Linear(8 * 6 * 6, 5),
            # Output (B, 5)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

    # Returns a epislon-greedy action
    def epsilon_action(
        self, epsilon: float, state: torch.Tensor, action_space: Space
    ) -> Number:
        r = random.random()
        if r < epsilon:
            return action_space.sample()

        with torch.no_grad():
            return self.forward(state.reshape((1, 3, 96, 96)).to(device)).argmax().item()

## Experience Replay

During tabular Q-learning, we learn by building a Q-table. This works because each state–action update is independent and does not directly interfere with other entries.

However, in a DQN, applying this principle directly would lead to unstable training, because the neural network generalises across states and actions. An update for one transition affects the Q-values of many other states, and consecutive samples are highly correlated. As a result, the network would need to learn from non-stationary and correlated data, which can cause divergence or oscillation.

This is mitigated using experience replay, where transitions are stored and randomly sampled during training, breaking temporal correlations and stabilising learning.

To start training, the experience replay buffer is first bootstrapped with an initial set of transitions collected by acting randomly in the environment.

In [9]:
Transition = namedtuple(
    "Transition", ("observation", "action", "reward", "next_observation", "done")
)

class ReplayBuffer:
    def __init__(self, capacity: int, batch_size: int):
        self.store = deque["Transition"]([], maxlen=capacity)
        self.batch_size = batch_size

    def get_batch(self):
        return random.sample(self.store, self.batch_size)

    def __len__(self):
        return len(self.store)

    def push(self, t: Transition):
        self.store.append(t)

    def bootstrap(self, env: gym.Env):
        state, _ = env.reset()

        observation = (torch.Tensor(state).permute(2, 0, 1) / 255.0).to(device)
        for _ in range(self.batch_size):
            random_action = env.action_space.sample()

            next_observation, reward, terminated, truncated, _ = env.step(random_action)
            next_observation = (torch.Tensor(next_observation).permute(2, 0, 1) / 255.0).to(device)

            t = Transition(
                observation,
                random_action,
                reward,
                next_observation,
                terminated or truncated,
            )

            self.push(t)

            if terminated or truncated:
                observation, _ = env.reset()
                observation = (torch.Tensor(observation).permute(2, 0, 1) / 255.0).to(device)
                continue

            observation = next_observation

        env.reset()

In [10]:
# Training Constants
GAMMA         = 0.99    # Discount factor (how far-sighted should the agent be?)

# Epsilon is the percentage of actions that should be random
# At the start of training, actions should be majority random to allow the agent to explore
# Towards the ends, the agent should only choose actions that increase the reward
EPSILON_START = 1.0    
EPSILON_END   = 0.05
EPSILON_DECAY = 100_000 # Steps required to reach EPSILON_DECAY

LEARNING_RATE = 3e-4

BATCH_SIZE    = 32 # How much transitions are in a batch?
REPLAY_BUFFER_SIZE         = 50_000 # Total number of transitions stored in the experience buffer at one time

EPISODES                   = 1500 # Number of episodes to train the agent
TARGET_UPDATE_SAMPLE_COUNT = 2500 # Number of steps needed to update the target network

LOSS_FN: torch.nn.MSELoss  = torch.nn.MSELoss()

def get_epsilon(step):
    return (
        EPSILON_END
        + (EPSILON_START - EPSILON_END)
        * torch.exp(torch.tensor(-1.0 * step / EPSILON_DECAY)).item()
    )

# Statistic Constants
RECORD_VIDEO_EPISODES = 200

In [11]:
env = gym.make("CarRacing-v3", render_mode="rgb_array", continuous=False, domain_randomize=False)
env = RecordVideo(
    env,
    video_folder="racing-training",
    name_prefix="training",
    episode_trigger=lambda x: x % RECORD_VIDEO_EPISODES == 0  # Only record every RECORD_VIDEO_EPISODES episodes
)
env = RecordEpisodeStatistics(env)

In [12]:
# Set all seeds to 0 for training reproducibility
random.seed(0)
torch.manual_seed(0)
state, info = env.reset(seed=0)
env.action_space.seed(0)
env.observation_space.seed(0)


0

In [ ]:
replay_buffer = ReplayBuffer(REPLAY_BUFFER_SIZE, BATCH_SIZE)

q_network = DQN().to(device)
target_network = DQN().to(device)

optimizer = torch.optim.AdamW(q_network.parameters(), lr=LEARNING_RATE, amsgrad=True)

target_network.load_state_dict(q_network.state_dict())

replay_buffer.bootstrap(env)

global_steps = 0

for e in range(EPISODES):
    episode_over = False
    episode_reward = 0
    observation, info = env.reset()
    observation = (torch.Tensor(observation).permute(2, 0, 1) / 255.0).to(device)    
    while not episode_over:
        global_steps += 1
        epsilon = get_epsilon(global_steps)

        # After TARGET_UPDATE_SAMPLE_COUNT, copy the q-network to the target network
        if global_steps % TARGET_UPDATE_SAMPLE_COUNT == 0:
            target_network.load_state_dict(q_network.state_dict())

        # Choose an epsilon-greedy action
        action = q_network.epsilon_action(epsilon, observation, env.action_space)

        # Make this action
        next_observation, reward, terminated, truncated, info = env.step(action)
        next_observation = (torch.Tensor(next_observation).permute(2, 0, 1) / 255.0).to(device)
        episode_reward += float(reward)
        episode_over = terminated or truncated

        # Store the transition in the Experience Replay
        t = Transition(
            observation, action, float(reward), next_observation, episode_over
        )
        replay_buffer.push(t)

        # Get a batch of transitions to train on
        batch = replay_buffer.get_batch()
        
        observations = torch.stack([b.observation for b in batch]).to(device)
        actions = torch.tensor([b.action for b in batch], dtype=torch.int).to(device)
        rewards = torch.Tensor([b.reward for b in batch]).to(device)
        next_observations = torch.stack([b.next_observation for b in batch]).to(device)
        episodes_over = torch.Tensor([b.done for b in batch]).to(device)

        q_network.train()

        # Compute q values and target q values
        q_values = q_network.forward(observations)[torch.arange(32), actions]
        with torch.no_grad():
            target_q_values = rewards + (
                GAMMA
                * target_network.forward(next_observations).max(1).values
                * (1 - episodes_over.to(device))
            )
        loss = LOSS_FN(q_values, target_q_values)

        # Train the q network
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_value_(q_network.parameters(), 10)
        optimizer.step()

        observation = next_observation

    print(
        f"Episode: {e} Reward: {episode_reward} Replay Buffer: {((float(len(replay_buffer)) / REPLAY_BUFFER_SIZE) * 100):.2f}%"
    )

In [9]:
torch.save(q_network.state_dict(), "model.pth")